![](image.jpg)


Dive into the heart of data science with a project that combines healthcare insights and predictive analytics. As a Data Scientist at a top Health Insurance company, you have the opportunity to predict customer healthcare costs using the power of machine learning. Your insights will help tailor services and guide customers in planning their healthcare expenses more effectively.

## Dataset Summary

Meet your primary tool: the `insurance.csv` dataset. Packed with information on health insurance customers, this dataset is your key to unlocking patterns in healthcare costs. Here's what you need to know about the data you'll be working with:

## insurance.csv
| Column    | Data Type | Description                                                      |
|-----------|-----------|------------------------------------------------------------------|
| `age`       | int       | Age of the primary beneficiary.                                  |
| `sex`       | object    | Gender of the insurance contractor (male or female).             |
| `bmi`       | float     | Body mass index, a key indicator of body fat based on height and weight. |
| `children`  | int       | Number of dependents covered by the insurance plan.              |
| `smoker`    | object    | Indicates whether the beneficiary smokes (yes or no).            |
| `region`    | object    | The beneficiary's residential area in the US, divided into four regions. |
| `charges`   | float     | Individual medical costs billed by health insurance.             |



A bit of data cleaning is key to ensure the dataset is ready for modeling. Once your model is built using the `insurance.csv` dataset, the next step is to apply it to the `validation_dataset.csv`. This new dataset, similar to your training data minus the `charges` column, tests your model's accuracy and real-world utility by predicting costs for new customers.

## Let's Get Started!

This project is your playground for applying data science in a meaningful way, offering insights that have real-world applications. Ready to explore the data and uncover insights that could revolutionize healthcare planning? Let's begin this exciting journey!

In [1]:
# ============================================================
# Project: From Data to Dollars — Predicting Insurance Charges
# ============================================================
# Goal:
#   Predict individual medical costs (charges) using demographic
#   and health features from insurance.csv.
#   Evaluate with R² and apply the model to validation_dataset.csv.
# ============================================================

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# ------------------------------------------------------------
# 1. Load the data
# ------------------------------------------------------------
insurance = pd.read_csv("insurance.csv")
insurance.head()


,age,sex,bmi,children,smoker,region,charges
0,19.0,female,27.900,0.0,yes,southwest,16884.924
1,18.0,male,33.770,1.0,no,Southeast,1725.5523
2,28.0,male,33.000,3.0,no,southeast,$4449.462
3,33.0,male,22.705,0.0,no,northwest,$21984.47061
4,32.0,male,28.880,0.0,no,northwest,$3866.8552


In [2]:
print(insurance.isnull().sum()) # چک کردن مقادیر گم‌شده

age         66
sex         66
bmi         66
children    66
smoker      66
region      66
charges     54
dtype: int64


In [3]:
# ------------------------------------------------------------
# 2. Quick exploration
# ------------------------------------------------------------
print("Shape:", insurance.shape)
print("\nMissing values per column:")
print(insurance.isnull().sum())
print("\nData types:")
print(insurance.dtypes)
print("\nSummary statistics:")
print(insurance.describe(include="all"))

# ------------------------------------------------------------
# 3. Preprocessing function
# ------------------------------------------------------------
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean and encode the insurance dataset so it is ready for modeling.

    Steps:
      - Drop rows with missing values.
      - Strip '$' from charges and cast to float.
      - Encode sex: male -> 0, female -> 1.
      - Encode smoker: no -> 0, yes -> 1.
      - Lowercase region and one-hot encode with drop_first=True.
      - Cast age and children to int.
    """
    df = df.copy()
    df = df.dropna()
    # Clean the target column when present
    if "charges" in df.columns:
        df["charges"] = (
            df["charges"]
            .astype(str)
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
            .astype(float)
        )

    # Binary encodings
    df["sex"] = df["sex"].map({"male": 0, "female": 1})
    df["smoker"] = df["smoker"].map({"no": 0, "yes": 1})

    # Region: lowercase + one-hot
    df["region"] = df["region"].str.lower()
    region_dummies = pd.get_dummies(
        df["region"], prefix="region", drop_first=True
    )
    df = pd.concat([df, region_dummies], axis=1)
    df = df.drop(columns="region")

    # Type casting
    df["age"] = df["age"].astype(int)
    df["children"] = df["children"].astype(int)
    df["bmi"] = df["bmi"].astype(float)

    df = df.dropna()
    return df


# Apply preprocessing to the training data
insurance_clean = preprocess(insurance)
insurance_clean.head()

Shape: (1338, 7)

Missing values per column:
age         66
sex         66
bmi         66
children    66
smoker      66
region      66
charges     54
dtype: int64

Data types:
age         float64
sex          object
bmi         float64
children    float64
smoker       object
region       object
charges      object
dtype: object

Summary statistics:
                age   sex          bmi     children smoker     region charges
count   1272.000000  1272  1272.000000  1272.000000   1272       1272    1284
unique          NaN     6          NaN          NaN      2          8    1272
top             NaN  male          NaN          NaN     no  Southeast    $nan
freq            NaN   517          NaN          NaN   1013        172      12
mean      35.214623   NaN    30.560550     0.948899    NaN        NaN     NaN
std       22.478251   NaN     6.095573     1.303532    NaN        NaN     NaN
min      -64.000000   NaN    15.960000    -4.000000    NaN        NaN     NaN
25%       24.750000   NaN

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,1.0,27.900,0,1,16884.92400,False,False,True
1,18,0.0,33.770,1,0,1725.55230,False,True,False
2,28,0.0,33.000,3,0,4449.46200,False,True,False
3,33,0.0,22.705,0,0,21984.47061,True,False,False
4,32,0.0,28.880,0,0,3866.85520,True,False,False


In [4]:
# ------------------------------------------------------------
# 4. Feature / target separation
# ------------------------------------------------------------
X = insurance_clean.drop(columns="charges")
y = insurance_clean["charges"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features:", list(X.columns))

# ------------------------------------------------------------
# 5. Train / test split
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


X shape: (973, 8)
y shape: (973,)
Features: ['age', 'sex', 'bmi', 'children', 'smoker', 'region_northwest', 'region_southeast', 'region_southwest']
X_train: (778, 8)
X_test : (195, 8)


In [5]:
# ------------------------------------------------------------
# 6. Train models
# ------------------------------------------------------------
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

rf_reg = RandomForestRegressor(
    n_estimators=300, random_state=42, n_jobs=-1
)
rf_reg.fit(X_train, y_train)

# ------------------------------------------------------------
# 7. Evaluate with R²
# ------------------------------------------------------------
lin_pred = lin_reg.predict(X_test)
rf_pred = rf_reg.predict(X_test)

lin_r2 = r2_score(y_test, lin_pred)
rf_r2 = r2_score(y_test, rf_pred)

print(f"Linear Regression R² : {lin_r2:.4f}")
print(f"Random Forest     R² : {rf_r2:.4f}")

# Select the best model (Linear Regression in this case is competitive
# and much more interpretable, so we keep it as the final model)
model = lin_reg
r2_score = lin_r2
print(f"\nFinal model: Linear Regression")
print(f"Final R²  : {r2_score:.4f}")


Linear Regression R² : 0.7298
Random Forest     R² : 0.8663

Final model: Linear Regression
Final R²  : 0.7298


In [6]:
# ------------------------------------------------------------
# 8. Inspect feature coefficients
# ------------------------------------------------------------
coefficients = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": model.coef_,
}).sort_values("coefficient", ascending=False)

print(coefficients)




            feature   coefficient
4            smoker  23632.575198
2               bmi    341.479007
3          children    318.396012
1               sex    144.620450
5  region_northwest    130.273930
0               age    112.029170
7  region_southwest     17.451136
6  region_southeast   -654.189509


In [7]:
# ------------------------------------------------------------
# 9. Apply the model to the validation dataset
# ------------------------------------------------------------
validation_data = pd.read_csv("validation_dataset.csv")
validation_data = preprocess(validation_data)

# Align columns exactly with the training features
validation_data = validation_data.reindex(
    columns=X_train.columns, fill_value=0
)

print("Validation shape after alignment:", validation_data.shape)
print(validation_data.head())

Validation shape after alignment: (50, 8)
   age  sex        bmi  children  smoker  region_northwest  region_southeast  \
0   18    1  24.090000         1       0             False              True   
1   39    0  26.410000         0       1             False             False   
2   27    0  29.150000         0       1             False              True   
3   71    0  65.502135        13       1             False              True   
4   28    0  38.060000         0       0             False              True   

   region_southwest  
0             False  
1             False  
2             False  
3             False  
4             False  


In [8]:

# ------------------------------------------------------------
# 10. Predict charges and enforce a minimum charge of 1000
# ------------------------------------------------------------
predicted_charges = model.predict(validation_data)
predicted_charges = np.maximum(predicted_charges, 1000)

validation_data["predicted_charges"] = predicted_charges
validation_data.head()

,age,sex,bmi,children,smoker,region_northwest,region_southeast,region_southwest,predicted_charges
0,18,1,24.090000,1,0,False,True,False,3699.660234
1,39,0,26.410000,0,1,False,False,False,30668.252341
2,27,0,29.150000,0,1,False,True,False,29605.365275
3,71,0,65.502135,13,1,False,True,False,51087.287972
4,28,0,38.060000,0,0,False,True,False,9127.397201
